In [1]:
# Imports
import pandas as pd
import numpy as np
from findata.equity_prices import get_equity_prices

# Display options for debugging
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

Functions implementing the stop-loss framework S(γ, δ, J) and utilities

In [2]:
from __future__ import annotations

def load_spy_tlt_prices(start_date: str, end_date: str) -> pd.DataFrame:
    """Load SPY and TLT daily Close prices between inclusive dates.
    Returns DataFrame with columns ['SPY','TLT'] and DatetimeIndex. Drops rows with any NaNs.
    """
    df = get_equity_prices(
        tickers=["SPY", "TLT"],
        start_date=start_date,
        end_date=end_date,
        fields=["Close"],
        frequency="1d",
        auto_adjust=True,
    )
    # When multiple tickers, get_equity_prices returns MultiIndex columns (field, ticker)
    if isinstance(df.columns, pd.MultiIndex):
        closes = df["Close"].copy()
    else:
        closes = df[["Close"]].copy()
        closes.columns = ["SPY"]  # Fallback, though MultiIndex is expected for multi-ticker
    closes = closes.loc[(closes.index >= pd.to_datetime(start_date)) & (closes.index <= pd.to_datetime(end_date))]
    closes = closes.sort_index()
    # Ensure only SPY, TLT in order and drop any incomplete rows
    closes = closes.reindex(columns=[c for c in ["SPY", "TLT"] if c in closes.columns])
    closes = closes.dropna(how="any")
    return closes


def make_simple_returns(prices: pd.DataFrame) -> pd.DataFrame:
    prices = prices.sort_index()
    rets = prices.pct_change().dropna(how="any")
    # Ensure columns order
    cols = [c for c in ["SPY", "TLT"] if c in rets.columns]
    return rets[cols]


def stoploss_signal(returns: pd.Series, J: int, gamma: float, delta: float, est_window: int = 252) -> pd.Series:
    """
    Implement S(γ, δ, J) per Kaminski & Lo (Definition 1):
    - Exit when J-day standardized cumulative return z_J < -gamma
    - Re-enter when 1-day standardized return z_1 >= delta
    State transitions use prior-day signals; trades occur on the next bar.
    """
    r = returns.sort_index().copy()
    # Rolling estimates
    mu = r.rolling(est_window, min_periods=est_window).mean()
    sigma = r.rolling(est_window, min_periods=est_window).std(ddof=0)
    sigma = sigma.replace(0.0, np.nan)

    # J-day cumulative (sum of simple returns as paper’s trigger approximation)
    cum_J = r.rolling(J, min_periods=J).sum()
    z_J = (cum_J - J * mu) / (sigma * np.sqrt(J))

    # 1-day standardized return
    z_1 = (r - mu) / sigma

    idx = r.index
    st = pd.Series(index=idx, dtype=float)

    # Initialize: start invested (st=1) from first day we have all necessary estimates
    start_i = max(mu.first_valid_index() or idx[0], sigma.first_valid_index() or idx[0], cum_J.first_valid_index() or idx[0])
    if start_i is None:
        return pd.Series(1.0, index=idx)

    started = False
    prev = 1.0  # start in
    for t in idx:
        if not started:
            if t >= start_i:
                started = True
            else:
                st[t] = np.nan
                continue
        # Decisions use prior day's z values
        t_prev = idx[idx.get_loc(t) - 1] if idx.get_loc(t) > 0 else None
        if t_prev is None:
            st[t] = prev
            continue
        prev_state = prev
        exit_cond = False
        reenter_cond = False
        zj_prev = z_J.get(t_prev, np.nan)
        z1_prev = z_1.get(t_prev, np.nan)
        if prev_state == 1 and pd.notna(zj_prev) and (zj_prev < -gamma):
            exit_cond = True
        if prev_state == 0 and pd.notna(z1_prev) and (z1_prev >= delta):
            reenter_cond = True
        if exit_cond:
            curr = 0.0
        elif reenter_cond:
            curr = 1.0
        else:
            curr = prev_state
        st[t] = curr
        prev = curr

    # Forward-fill from first valid value; where NaN before start, set to 1 (invested)
    st = st.fillna(method="ffill")
    st = st.fillna(1.0)
    st.name = "st"
    return st.astype(float)


def make_asset_weights(st: pd.Series, rets: pd.DataFrame) -> pd.DataFrame:
    # Align index and lag by 1 day: weights on day t use state from t-1
    st_aligned = st.reindex(rets.index).shift(1)
    # For the first valid day, assume invested in SPY if missing
    if st_aligned.first_valid_index() is not None and np.isnan(st_aligned.iloc[0]):
        st_aligned.iloc[0] = 1.0
    st_aligned = st_aligned.fillna(method="ffill").clip(0.0, 1.0)
    w = pd.DataFrame({"SPY": st_aligned, "TLT": 1.0 - st_aligned}, index=rets.index)
    return w


def pass_through_returns(rets: pd.DataFrame) -> pd.DataFrame:
    rets = rets.sort_index()
    cols = [c for c in ["SPY", "TLT"] if c in rets.columns]
    return rets[cols]

Parameters and data loading

In [3]:
# Date range constrained by SPY/TLT overlap and the paper's sample end
START_DATE = "2002-07-26"
END_DATE   = "2011-11-07"

# Stop-loss parameters reflecting longer-term window as in the paper's key findings
J = 60          # trading days (~3 months)
GAMMA = 1.0     # exit threshold in z-score units
DELTA = 0.0     # re-entry threshold in z-score units
EST_WINDOW = 252  # rolling window for mean/std estimation

prices = load_spy_tlt_prices(START_DATE, END_DATE)
prices.head()

Ticker,SPY,TLT
Date,,
2002-07-30,58.842705,36.387642
2002-07-31,58.985073,36.838478
2002-08-01,57.445068,37.048279
2002-08-02,56.157421,37.427681
2002-08-05,54.203354,37.592834


In [4]:
# Compute per-asset simple daily returns
asset_returns = make_simple_returns(prices)
asset_returns.tail()

Ticker,SPY,TLT
Date,,
2011-10-31,-0.024106,0.039656
2011-11-01,-0.027888,0.033474
2011-11-02,0.016311,-0.012474
2011-11-03,0.018228,-0.013734
2011-11-04,-0.006099,0.001203


In [5]:
# Build stop-loss state on SPY; map to portfolio weights between SPY and TLT
st = stoploss_signal(asset_returns["SPY"], J=J, gamma=GAMMA, delta=DELTA, est_window=EST_WINDOW)
asset_weights = make_asset_weights(st, asset_returns)

# Sanity checks for required final DataFrames
assert isinstance(asset_weights, pd.DataFrame) and set(asset_weights.columns) == {"SPY","TLT"}
assert isinstance(asset_returns, pd.DataFrame) and set(asset_returns.columns) == {"SPY","TLT"}

asset_weights.tail()

/var/folders/jr/prby656n267_0pztcwybgy5r0000gn/T/ipykernel_6999/688721664.py:98: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  st = st.fillna(method="ffill")
/var/folders/jr/prby656n267_0pztcwybgy5r0000gn/T/ipykernel_6999/688721664.py:110: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  st_aligned = st_aligned.fillna(method="ffill").clip(0.0, 1.0)


,SPY,TLT
Date,,
2011-10-31,1.0,0.0
2011-11-01,1.0,0.0
2011-11-02,1.0,0.0
2011-11-03,1.0,0.0
2011-11-04,1.0,0.0


Final backtest call (must be defined by the user/environment): backtest(asset_weights, asset_returns)

In [6]:
# Exactly once backtest invocation, if available in environment
try:
    results = backtest(asset_weights, asset_returns)
    results
except NameError:
    print("User-defined function backtest(asset_weights, asset_returns) not found; skipping final backtest run.")

User-defined function backtest(asset_weights, asset_returns) not found; skipping final backtest run.
